# Scraper completo registroimprese.it (card + dettaglio, HTML + CSV)

Basato sulla logica di ricerca/paginazione gia' verificata (non si blocca):
`do_search` naviga con `driver.get()`, `go_to_next_page` clicca il link
"next" via AJAX e verifica il cambio pagina con un fingerprint prima di
proseguire.

Per ogni azienda trovata nei risultati:
1. Estrae i dati della card (ragione sociale, codice fiscale, comune,
   tipo, codice ATECO, classi, natura giuridica, costituzione, tag, sito).
2. Clicca sul nome azienda per aprire la pagina di dettaglio ("DATI
   ISCRITTI NEL REGISTRO IMPRESE"), salva l'HTML grezzo in
   `data/raw_dettaglio/<codice_fiscale>.html` ed estrae: url, settore,
   forma giuridica, sito internet, attivita' svolta, titoli/esperienze
   soci, relazioni professionali, diritti di proprieta' intellettuale
   (questi ultimi quattro letti da id di elementi HTML fissi, verificati
   con DevTools su un'azienda reale — non dal matching di testo, che e'
   fragile).
3. Torna ai risultati (prova "indietro" del browser; se non torna alla
   pagina giusta, rifa' la ricerca e riavanza fino alla pagina corrente:
   piu' lento ma sempre valido) e passa alla card successiva.

Tutto scritto in `data/companies.csv` (una riga per azienda, con TUTTI i
campi sopra). Ripartibile: le aziende gia' presenti (per codice fiscale)
vengono saltate, quindi puoi interrompere e rilanciare senza perdere nulla.
Gli errori non bloccano il resto: vengono loggati in
`data/errori_estrazione.csv`.


In [1]:
import csv
import math
import re
import time
import traceback
from pathlib import Path

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
    WebDriverException,
)
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait


In [2]:
# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #
SEARCH_URL = "https://startup.registroimprese.it/isin/search"

HERE = Path.cwd()  # __file__ non esiste in un notebook

DATA_DIR = HERE / "data"
RAW_DIR = DATA_DIR / "raw"                  # HTML/testo delle CARD
RAW_DETTAGLIO_DIR = DATA_DIR / "raw_dettaglio"  # HTML delle pagine di DETTAGLIO
CSV_PATH = DATA_DIR / "companies.csv"                 # SOLO campi della card
DETAIL_CSV_PATH = DATA_DIR / "dettagli_estratti.csv"   # SOLO campi del dettaglio
ERRORI_CSV = DATA_DIR / "errori_estrazione.csv"
TOGGLE_CSV_PATH = DATA_DIR / "toggle_flags.csv"  # SOLO i toggle, catturati live (non recuperabili da HTML salvato)
KEYWORDS_FILE = HERE / "keywords.txt"
CF_FILE = HERE / "codici_fiscali.txt"  # un codice fiscale per riga, per la modalita' "cerca per CF"

FIELD_TIMEOUT = 45   # secondi di attesa per campo di ricerca / card / dettaglio
BACK_TIMEOUT = 12    # secondi di attesa per il tentativo "indietro" dopo un dettaglio:
                     # se funziona di solito e' veloce; se non funziona meglio scoprirlo
                     # presto e passare al fallback (rifare la ricerca) invece di aspettare
                     # fino a FIELD_TIMEOUT per niente
SETTLE = 1.2         # pausa dopo un aggiornamento pagina/AJAX

# Campi della card di ricerca (logica originale, verificata) -> companies.csv
CSV_FIELDS = [
    "keyword",
    "ragione_sociale",
    "codice_fiscale",
    "provincia",
    "comune",
    "tipo",
    "codice_ateco",
    "classe_valore_produzione",
    "classe_addetti",
    "classe_capitale",
    "natura_giuridica",
    "costituzione",
    "sezione_data",
    "tag",
    "sito_web",
]

# Campi della pagina di dettaglio -> dettagli_estratti.csv (file separato)
DETAIL_CSV_FIELDS = [
    "keyword",
    "ragione_sociale",
    "codice_fiscale",
    "url",
    "aggiornamento_al",
    "impresa_costituita",
    "settore",
    "forma_giuridica",
    "sito_internet",
    "codice_ateco",
    "comune",
    "classe_produzione",
    "classe_addetti",
    "classe_capitale",
    "prevalenza_femminile",
    "prevalenza_giovanile",
    "prevalenza_straniera",
    "attivita_svolta",
    "titoli_esperienze",
    "relazioni_professionali",
    "diritti_proprieta_intellettuale",
    "requisiti_innovazione",
    "possesso_titoli_ip",
    "stadio_startup",
    "presentazione",
    "prodotto_servizio_stadio",
    "prodotto_servizio_testo",
    "team_stadio",
    "area_geografica_interesse",
    "canali_vendita",
    "business_model",
    "concorrenza",
    "innovazione_testo",
    "finanza",
    "interessi",
    "incubatore_accelerato",
    "dichiarazione_firma_digitale",
    "percentuale",
]

ERRORI_FIELDS = ["keyword", "pagina", "posizione", "ragione_sociale", "codice_fiscale", "errore"]

TOGGLE_CSV_FIELDS = [
    "codice_fiscale",
    "requisiti_innovazione",
    "possesso_titoli_ip",
    "incubatore_accelerato",
    "canali_vendita",
    "interessi",
]

# Etichette italiane sulla card, usate per estrarre i valori dal testo piatto.
CARD_LABELS = [
    "Codice fiscale",
    "Natura giuridica",
    "Comune",
    "DATI DI ISCRIZIONE:",
    "Costituzione Impresa",
    "Sezione Startup",
    "Sezione PMI",
    "Sezione",
    "Codice Ateco",
    "Classe Valore della Produzione",
    "Classe di Addetti",
    "Classe di Capitale",
]

# Etichette sulla pagina di dettaglio (per i campi in cima, riga singola).
DETAIL_TOP_LABELS = [
    "Aggiornamento al",
    "Denominazione",
    "Comune",
    "Codice fiscale",
    "Forma Giuridica",
    "Impresa costituita",
    "Sito internet",
    "Codice Ateco",
    "Settore",
]

# Id HTML fissi (verificati con DevTools su un'azienda reale) dove vive il
# testo COMPLETO di ogni sezione lunga, indipendentemente dal click su
# "Mostra tutto" (che cambia solo lo stile CSS, il testo e' gia' nel DOM).
SECTION_IDS = {
    "attivita_svolta": "mostPresentazioneAttivitaRi",
    "titoli_esperienze": "mostTeamRi",
    "relazioni_professionali": "mostFinanzaIncubatoriRi",
    "diritti_proprieta_intellettuale": "mostInnovazioneRi",
}


In [3]:
# --------------------------------------------------------------------------- #
# Browser
# --------------------------------------------------------------------------- #
def build_driver(headless: bool = True) -> webdriver.Chrome:
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,1100")
    opts.add_argument("--lang=it-IT")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(options=opts)
    driver.set_page_load_timeout(120)
    try:
        driver.execute_cdp_cmd(
            "Page.addScriptToEvaluateOnNewDocument",
            {"source": "Object.defineProperty(navigator,'webdriver',{get:()=>undefined})"},
        )
    except Exception:
        pass
    return driver


def wait_for_search_field(driver):
    '''Attende che la challenge JS TSPD finisca e appaia il campo di ricerca.'''
    WebDriverWait(driver, FIELD_TIMEOUT).until(
        lambda d: d.find_elements(By.NAME, "parolaChiaveFld")
    )


def ensure_italian(driver):
    '''L'header mostra un link 'ITA' solo quando l'interfaccia e' in inglese.'''
    try:
        ita = [a for a in driver.find_elements(By.XPATH, "//a")
               if a.text.strip().upper() == "ITA" and a.is_displayed()]
        if ita:
            driver.execute_script("arguments[0].click();", ita[0])
            time.sleep(SETTLE)
            wait_for_search_field(driver)
    except WebDriverException:
        pass


In [4]:
# --------------------------------------------------------------------------- #
# Ricerca (logica originale, verificata: non si blocca)
# --------------------------------------------------------------------------- #
def salva_diagnostica(driver, nome: str):
    '''Salva screenshot + HTML della pagina corrente, utile per capire
    cosa mostra davvero il sito quando qualcosa va storto (CAPTCHA, pagina
    di blocco, sfida anti-bot che non si risolve, ecc.).'''
    (DATA_DIR).mkdir(parents=True, exist_ok=True)
    safe = re.sub(r"[^A-Za-z0-9_-]", "_", nome)
    try:
        driver.save_screenshot(str(DATA_DIR / f"diagnostica_{safe}.png"))
    except WebDriverException:
        pass
    try:
        (DATA_DIR / f"diagnostica_{safe}.html").write_text(driver.page_source, encoding="utf-8")
    except WebDriverException:
        pass
    print(f"  [diagnostica] salvata in data/diagnostica_{safe}.png / .html")


def do_search(driver, keyword: str, tentativi: int = 3):
    '''Prova la ricerca fino a 'tentativi' volte, con attesa via via piu'
    lunga: la sfida anti-bot a volte impiega piu' del previsto a risolversi,
    specialmente dopo molte richieste ravvicinate.'''
    ultimo_errore = None
    for tentativo in range(1, tentativi + 1):
        try:
            driver.get(SEARCH_URL)
            timeout = FIELD_TIMEOUT * tentativo  # 45s, poi 90s, poi 135s...
            WebDriverWait(driver, timeout).until(
                lambda d: d.find_elements(By.NAME, "parolaChiaveFld")
            )
            ensure_italian(driver)

            box = driver.find_element(By.NAME, "parolaChiaveFld")
            box.clear()
            box.send_keys(keyword)
            time.sleep(0.4)
            btn = driver.find_element(By.NAME, "searchBtn")
            driver.execute_script("arguments[0].click();", btn)

            try:
                WebDriverWait(driver, timeout).until(
                    lambda d: get_cards(d) or _no_results(d)
                )
            except TimeoutException:
                pass
            time.sleep(SETTLE)
            return
        except TimeoutException as e:
            ultimo_errore = e
            print(f"  [do_search] tentativo {tentativo}/{tentativi} fallito (timeout {FIELD_TIMEOUT * tentativo}s)")
            if tentativo == tentativi:
                salva_diagnostica(driver, f"ricerca_{keyword}")
            else:
                time.sleep(3 * tentativo)  # pausa crescente prima di riprovare
    raise ultimo_errore


def _no_results(driver) -> bool:
    txt = driver.find_element(By.TAG_NAME, "body").text.lower()
    return "nessun risultato" in txt or "0 di 0" in txt


def total_results(driver) -> int:
    '''Estrae 'visualizzati 10 di 2601' -> 2601.'''
    txt = driver.find_element(By.TAG_NAME, "body").text
    m = re.search(r"di\s+([\d.\s]+)", txt)
    if m:
        return int(re.sub(r"[.\s]", "", m.group(1)))
    return 0


In [5]:
# --------------------------------------------------------------------------- #
# Card (logica originale, verificata)
# --------------------------------------------------------------------------- #
def get_cards(driver):
    return driver.find_elements(By.CSS_SELECTOR, "div.searchCompanyCard")


def first_card_fingerprint(driver) -> str:
    cards = get_cards(driver)
    if not cards:
        return ""
    try:
        return (cards[0].text or "")[:60]
    except StaleElementReferenceException:
        return ""


def _value_after_label(text: str, label: str, labels=None) -> str:
    labels = labels if labels is not None else CARD_LABELS
    idx = text.find(label)
    if idx == -1:
        return ""
    rest = text[idx + len(label):]
    end = len(rest)
    for other in labels:
        if other == label:
            continue
        pos = rest.find(other)
        if pos != -1:
            end = min(end, pos)
    value = rest[:end].strip(" :\n\t")
    return value.splitlines()[0].strip() if value else ""


def get_card_identity(card):
    '''Ragione sociale e codice fiscale, per matching/nomi file.'''
    ragione = ""
    try:
        ragione = card.find_element(By.CSS_SELECTOR, "a.link").text.strip()
    except NoSuchElementException:
        pass
    cf = ""
    m = re.search(r"Codice fiscale\s*\n?\s*(\w+)", card.text)
    if m:
        cf = m.group(1)
    return ragione, cf


def parse_card(card, keyword: str):
    text = card.text

    ragione = ""
    try:
        ragione = card.find_element(By.CSS_SELECTOR, "a.link").text.strip()
    except NoSuchElementException:
        pass

    sito = ""
    for a in card.find_elements(By.CSS_SELECTOR, "a[href^='http']"):
        href = a.get_attribute("href") or ""
        if "registroimprese.it" not in href:
            sito = href
            break

    tags = [t.text.strip() for t in
            card.find_elements(By.CSS_SELECTOR, "div.tag.label, div.tag")
            if t.text.strip()]
    tag_str = " | ".join(dict.fromkeys(tags))

    cf = _value_after_label(text, "Codice fiscale")
    comune = _value_after_label(text, "Comune")
    prov = ""
    m = re.search(r"\(([A-Z]{2})\)", comune)
    if m:
        prov = m.group(1)

    if "Sezione Startup" in text:
        tipo = "Startup"
    elif "Sezione PMI" in text:
        tipo = "PMI innovativa"
    else:
        tipo = ""

    sezione_data = ""
    m = re.search(r"Sezione\s+\w+\s*\n?\s*([0-3]?\d/[01]?\d/\d{4})", text)
    if m:
        sezione_data = m.group(1)

    costituzione = _value_after_label(text, "Costituzione Impresa")
    m = re.search(r"([0-3]?\d/[01]?\d/\d{4})", costituzione)
    if m:
        costituzione = m.group(1)

    row = {
        "keyword": keyword,
        "ragione_sociale": ragione,
        "codice_fiscale": cf,
        "provincia": prov,
        "comune": comune,
        "tipo": tipo,
        "codice_ateco": _value_after_label(text, "Codice Ateco"),
        "classe_valore_produzione": _value_after_label(text, "Classe Valore della Produzione"),
        "classe_addetti": _value_after_label(text, "Classe di Addetti"),
        "classe_capitale": _value_after_label(text, "Classe di Capitale"),
        "natura_giuridica": _value_after_label(text, "Natura giuridica"),
        "costituzione": costituzione,
        "sezione_data": sezione_data,
        "tag": tag_str,
        "sito_web": sito,
    }
    return row, text


In [6]:
# --------------------------------------------------------------------------- #
# Paginazione (logica originale, verificata: non si blocca)
# --------------------------------------------------------------------------- #
def go_to_next_page(driver) -> bool:
    '''Clicca il link 'next' del navigatore. Ritorna False sull'ultima pagina.'''
    before = first_card_fingerprint(driver)
    links = [a for a in driver.find_elements(By.XPATH, "//a[contains(@href,'-next')]")
             if a.is_displayed()]
    if not links:
        return False
    try:
        driver.execute_script("arguments[0].click();", links[0])
    except WebDriverException:
        return False
    try:
        WebDriverWait(driver, FIELD_TIMEOUT).until(
            lambda d: get_cards(d) and first_card_fingerprint(d) != before
        )
    except TimeoutException:
        return False
    time.sleep(SETTLE)
    return True


In [7]:
# --------------------------------------------------------------------------- #
# Pagina di dettaglio: apertura via click + estrazione (validate su MANTA
# AIRCRAFT S.R.L.). Il link sul nome azienda ha href="javascript:;" (non
# naviga da solo), va cliccato davvero.
# --------------------------------------------------------------------------- #
def wait_for_detail_page(driver, timeout=FIELD_TIMEOUT):
    WebDriverWait(driver, timeout).until(
        lambda d: "DATI ISCRITTI NEL REGISTRO IMPRESE" in d.find_element(By.TAG_NAME, "body").text
    )


def expand_all_sections(driver):
    '''Clicca 'Mostra tutto' (non indispensabile: il testo e' gia' nel DOM,
    ma innocuo tenerlo come rete di sicurezza).'''
    links = driver.find_elements(By.XPATH, "//*[contains(text(),'Mostra tutto')]")
    for l in links:
        try:
            driver.execute_script("arguments[0].click();", l)
        except WebDriverException:
            pass
    if links:
        time.sleep(0.2)


def click_apri_dettaglio(driver, card):
    '''Clicca sul nome azienda nella card. Ritorna (html, url,
    opened_new_tab, toggle_flags) oppure (None, None, False, {}) se il click
    non produce nulla entro il timeout.'''
    try:
        link = card.find_element(By.CSS_SELECTOR, "a.link")
    except NoSuchElementException:
        return None, None, False, {}

    handles_before = driver.window_handles
    try:
        driver.execute_script("arguments[0].click();", link)
    except WebDriverException:
        return None, None, False, {}

    try:
        WebDriverWait(driver, FIELD_TIMEOUT).until(
            lambda d: len(d.window_handles) > len(handles_before)
            or "DATI ISCRITTI NEL REGISTRO IMPRESE" in d.find_element(By.TAG_NAME, "body").text
        )
    except TimeoutException:
        return None, None, False, {}

    opened_new_tab = len(driver.window_handles) > len(handles_before)
    if opened_new_tab:
        new_handle = [h for h in driver.window_handles if h not in handles_before][0]
        driver.switch_to.window(new_handle)
        try:
            wait_for_detail_page(driver)
        except TimeoutException:
            pass

    expand_all_sections(driver)
    html = driver.page_source
    url = driver.current_url
    # I toggle (interruttori blu/grigi) vanno letti ORA: sono immagini generate
    # dal server, il loro stato non e' nel testo/HTML salvato, quindi va fatto
    # mentre la pagina di dettaglio e' ancora viva nel browser.
    toggle_flags = estrai_toggle_flags(driver)

    if opened_new_tab:
        driver.close()
        driver.switch_to.window(handles_before[0])

    return html, url, opened_new_tab, toggle_flags


def torna_ai_risultati(driver, fingerprint_attesa: str, keyword: str, pagina: int) -> bool:
    '''Dopo un dettaglio aperto NELLA STESSA scheda, prova 'indietro'; se
    entro il timeout i risultati non tornano com'erano, rifa' la ricerca
    (do_search, gia' verificata affidabile) e riavanza fino a 'pagina' con
    go_to_next_page (gia' verificata affidabile). Ritorna True se
    'indietro' ha funzionato al volo.'''
    try:
        driver.back()
        time.sleep(0.6)  # da' al browser il tempo di iniziare davvero la
                          # navigazione prima di controllare il fingerprint,
                          # altrimenti si rischia di leggere ancora la
                          # pagina di dettaglio (o uno stato intermedio)
        WebDriverWait(driver, BACK_TIMEOUT).until(
            lambda d: first_card_fingerprint(d) == fingerprint_attesa
        )
        return True
    except (TimeoutException, WebDriverException):
        pass

    try:
        do_search(driver, keyword)
    except Exception as e:
        # Anche se la ricerca di recupero fallisce del tutto, non blocchiamo
        # lo scraper: torniamo False e la card/keyword successiva ripartira'
        # comunque da una nuova ricerca.
        print(f"    [torna_ai_risultati] anche la ricerca di recupero e' fallita: {e}")
        return False

    for _ in range(pagina - 1):
        if not go_to_next_page(driver):
            break
    return False


def _testo_da_id(soup, element_id: str) -> str:
    div = soup.find(id=element_id)
    if div is None:
        return ""
    testo = div.get_text("\n")
    righe = [l.strip() for l in testo.splitlines() if l.strip()]
    return "\n".join(righe)


# --------------------------------------------------------------------------- #
# Campi testuali aggiuntivi del dettaglio (letti dall'HTML salvato, non
# servono screenshot). Validati su JUMBLEBIT SRL.
# --------------------------------------------------------------------------- #
def _classe_da_testo(text: str, nome: str) -> str:
    '''"classe di produzione/addetti/capitale" -> "1-100K euro | A".'''
    m = re.search(rf"classe di {nome}\s*\n\s*(.+?)\s*\n\s*(\S+)", text)
    return f"{m.group(1).strip()} | {m.group(2).strip()}" if m else ""


def _prevalenza_da_testo(text: str, nome: str) -> str:
    '''"prevalenza femminile/giovanile/straniera" -> "Esclusiva" / "NO" / ecc.'''
    m = re.search(rf"prevalenza {nome}\s*\n\s*(\S.*?)\s*\n", text)
    return m.group(1).strip() if m else ""


def _step_attivo(soup, anchor_id: str) -> str:
    '''Etichetta dello step con class "active" dentro la sezione con
    id=anchor_id (es. stadioAnchor, prodottoAnchor, teamAnchor).'''
    sezione = soup.find(id=anchor_id)
    if sezione is None:
        return ""
    step = sezione.find(class_="step active")
    if step is None:
        return ""
    label = step.find("label")
    return label.get_text(strip=True) if label else ""


def _testo_sezione(soup, header_text: str) -> str:
    '''Cerca un <h4> il cui testo e' esattamente header_text e ritorna il
    testo del blocco "twocolumntext" successivo (PRESENTAZIONE, BUSINESS
    MODEL, CONCORRENZA, INNOVAZIONE...).'''
    h4 = soup.find(lambda t: t.name == "h4" and t.get_text(strip=True) == header_text)
    if h4 is None:
        return ""
    div = h4.find_next("div", class_="twocolumntext")
    return " ".join(div.get_text(" ", strip=True).split()) if div else ""


def _prima_twocolumntext(soup, anchor_id: str) -> str:
    '''Testo descrittivo dentro una sezione con anchor (usato per
    PRODOTTO/SERVIZIO_TESTO, che non ha un'intestazione propria).'''
    sezione = soup.find(id=anchor_id)
    if sezione is None:
        return ""
    div = sezione.find("div", class_="twocolumntext")
    return " ".join(div.get_text(" ", strip=True).split()) if div else ""


def _area_geografica(soup) -> str:
    '''Elenco delle aree geografiche di interesse, unite da " | ".'''
    h4 = soup.find(lambda t: t.name == "h4" and "AREA GEOGRAFICA" in t.get_text().upper())
    if h4 is None:
        return ""
    cont = h4.find_next(id=re.compile(r"^mCSB_\d+_container$"))
    if cont is None:
        return ""
    parti = [s.strip() for s in cont.stripped_strings]
    return " | ".join(dict.fromkeys(p for p in parti if p))


def _testo_finanza(soup) -> str:
    h4 = soup.find(lambda t: t.name == "h4" and t.get_text(strip=True) == "FINANZA")
    if h4 is None:
        return ""
    container = h4.find_parent("div")
    span = container.find_next("span") if container else None
    return " ".join(span.get_text(" ", strip=True).split()) if span else ""


def _testo_firma_digitale(soup) -> str:
    h3 = soup.find(lambda t: t.name == "h3" and "FIRMA DIGITALE" in t.get_text())
    if h3 is None:
        return ""
    span = h3.find("span")
    return " ".join(span.get_text(" ", strip=True).split()) if span else ""


def _percentuale(soup) -> str:
    '''Il cerchio percentuale (widget "circliful") ha l'informazione gia'
    pronta negli attributi data-*, non serve leggere pixel.'''
    el = soup.find(class_="circliful")
    if el is None:
        return ""
    testo = el.get("data-text")
    if testo:
        return testo.strip()
    percento = el.get("data-percent")
    return f"{percento}%" if percento else ""


def estrai_campi_dettaglio(html: str) -> dict:
    '''Estrae i campi TESTUALI della pagina di dettaglio da un HTML gia'
    scaricato. Le 4 sezioni lunghe usano gli id fissi verificati
    (SECTION_IDS); i campi in cima usano il matching per etichette. I
    campi "a interruttore" (SI/NO/elenco selezionati) NON sono qui: vanno
    letti dal browser live via estrai_toggle_flags().'''
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")

    fields = {
        "aggiornamento_al": _value_after_label(text, "Aggiornamento al", DETAIL_TOP_LABELS),
        "impresa_costituita": _value_after_label(text, "Impresa costituita", DETAIL_TOP_LABELS),
        "settore": _value_after_label(text, "Settore", DETAIL_TOP_LABELS),
        "forma_giuridica": _value_after_label(text, "Forma Giuridica", DETAIL_TOP_LABELS),
        "sito_internet": _value_after_label(text, "Sito internet", DETAIL_TOP_LABELS),
        "codice_ateco": _value_after_label(text, "Codice Ateco", DETAIL_TOP_LABELS),
        "comune": _value_after_label(text, "Comune", DETAIL_TOP_LABELS),
        "classe_produzione": _classe_da_testo(text, "produzione"),
        "classe_addetti": _classe_da_testo(text, "addetti"),
        "classe_capitale": _classe_da_testo(text, "capitale"),
        "prevalenza_femminile": _prevalenza_da_testo(text, "femminile"),
        "prevalenza_giovanile": _prevalenza_da_testo(text, "giovanile"),
        "prevalenza_straniera": _prevalenza_da_testo(text, "straniera"),
        "stadio_startup": _step_attivo(soup, "stadioAnchor"),
        "presentazione": _testo_sezione(soup, "PRESENTAZIONE"),
        "prodotto_servizio_stadio": _step_attivo(soup, "prodottoAnchor"),
        "prodotto_servizio_testo": _prima_twocolumntext(soup, "prodottoAnchor"),
        "team_stadio": _step_attivo(soup, "teamAnchor"),
        "area_geografica_interesse": _area_geografica(soup),
        "business_model": _testo_sezione(soup, "BUSINESS MODEL E PROVENIENZA DEI PROFITTI"),
        "concorrenza": _testo_sezione(soup, "CONCORRENZA"),
        "innovazione_testo": _testo_sezione(soup, "INNOVAZIONE"),
        "finanza": _testo_finanza(soup),
        "dichiarazione_firma_digitale": _testo_firma_digitale(soup),
        "percentuale": _percentuale(soup),
    }
    for campo, elem_id in SECTION_IDS.items():
        fields[campo] = _testo_da_id(soup, elem_id)
    return fields


# --------------------------------------------------------------------------- #
# Campi "a interruttore" (toggle blu=SI / grigio=NO). Lo stato NON e' nel
# testo/HTML: e' disegnato dal server dentro un'immagine. Prima lo leggevamo
# con ~40 comandi Selenium separati (uno screenshot per immagine): troppo
# lento, rallentava la pagina di dettaglio e faceva scattare piu' spesso la
# sfida anti-bot al ritorno ai risultati. Ora lo facciamo con UNA sola
# chiamata JS eseguita nel browser (execute_script): ogni immagine viene
# disegnata su un canvas e se ne calcola il colore medio li', tutto in un
# colpo solo, senza andare avanti e indietro con Selenium.
# Soglie di colore tarate a occhio: da verificare/aggiustare sul primo run
# reale (INSPECT=True), confrontando l'output con quello che si vede a
# schermo.
# --------------------------------------------------------------------------- #
_JS_TOGGLE_FLAGS = """
function avgColor(img) {
    try {
        var w = img.naturalWidth || img.width;
        var h = img.naturalHeight || img.height;
        if (!w || !h) return null;
        var canvas = document.createElement('canvas');
        canvas.width = w; canvas.height = h;
        var ctx = canvas.getContext('2d');
        ctx.drawImage(img, 0, 0, w, h);
        var data = ctx.getImageData(0, 0, w, h).data;
        var r = 0, g = 0, b = 0, n = 0;
        for (var i = 0; i < data.length; i += 4) {
            if (data[i + 3] === 0) continue;  // pixel trasparente, salta
            r += data[i]; g += data[i + 1]; b += data[i + 2]; n++;
        }
        if (!n) return null;
        return [r / n, g / n, b / n];
    } catch (e) {
        return null;
    }
}
function isBlu(img) {
    var c = avgColor(img);
    if (!c) return false;
    return (c[2] - c[0]) > 25 && (c[2] - c[1]) > 15;
}
function findImgBySrc(substr) {
    var imgs = document.querySelectorAll('img');
    for (var i = 0; i < imgs.length; i++) {
        if (imgs[i].src && imgs[i].src.indexOf(substr) !== -1) return imgs[i];
    }
    return null;
}
function flagSingolo(substr) {
    var img = findImgBySrc(substr);
    if (!img) return "";
    return isBlu(img) ? "SI" : "NO";
}
function sezioneMultiAttive(headerText) {
    var h4s = document.querySelectorAll('h4');
    var header = null;
    for (var i = 0; i < h4s.length; i++) {
        if (h4s[i].textContent.trim() === headerText) { header = h4s[i]; break; }
    }
    if (!header || !header.parentElement) return "";
    var container = header.parentElement.nextElementSibling;
    if (!container) return "";
    var cols = container.querySelectorAll('div.column');
    var attive = [];
    for (var i = 0; i < cols.length; i++) {
        var span = cols[i].querySelector('span');
        var img = cols[i].querySelector('img');
        if (!span || !img) continue;
        var label = span.textContent.trim();
        if (label && isBlu(img)) attive.push(label);
    }
    return attive.join(' | ');
}

var requisiti = [];
if (flagSingolo('imgRicercaSviluppo') === 'SI') requisiti.push('R&S');
if (flagSingolo('imgLaureatiMagistrali') === 'SI') requisiti.push('Team Qualificato');
if (flagSingolo('imgBrevetti') === 'SI') requisiti.push('Proprietà Intellettuale');

return {
    requisiti_innovazione: requisiti.join(' | '),
    possesso_titoli_ip: flagSingolo('imgPossessoTitoliIntellettuali'),
    incubatore_accelerato: flagSingolo('imgSelettoreIncubatoreAccelerato'),
    canali_vendita: sezioneMultiAttive('CANALI DI VENDITA'),
    interessi: sezioneMultiAttive('INTERESSI')
};
"""


def estrai_toggle_flags(driver) -> dict:
    '''Legge tutti i toggle in UNA sola chiamata JS lato browser (invece
    di ~40 comandi Selenium separati): molto piu' veloce, non rallenta la
    pagina di dettaglio prima del ritorno ai risultati. Va chiamata MENTRE
    la pagina di dettaglio e' ancora aperta nel browser (viene gia' fatto
    dentro click_apri_dettaglio). Ritorna {} se qualcosa va storto, senza
    bloccare il resto dello scraping.'''
    try:
        risultato = driver.execute_script(_JS_TOGGLE_FLAGS)
    except WebDriverException:
        return {}
    return risultato or {}


In [8]:
# --------------------------------------------------------------------------- #
# Persistenza (ripartibile: salta i codici fiscali gia' presenti)
# --------------------------------------------------------------------------- #
def load_seen() -> set:
    seen = set()
    if CSV_PATH.exists():
        with CSV_PATH.open(newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f, delimiter=";"):
                if r.get("codice_fiscale"):
                    seen.add(r["codice_fiscale"])
    return seen


def ensure_csv_header():
    if not CSV_PATH.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS, delimiter=";").writeheader()


def append_row(row: dict):
    with CSV_PATH.open("a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS, delimiter=";").writerow(
            {k: row.get(k, "") for k in CSV_FIELDS}
        )


def ensure_detail_csv_header():
    if not DETAIL_CSV_PATH.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with DETAIL_CSV_PATH.open("w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=DETAIL_CSV_FIELDS, delimiter=";").writeheader()


def append_detail_row(row: dict):
    with DETAIL_CSV_PATH.open("a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=DETAIL_CSV_FIELDS, delimiter=";").writerow(
            {k: row.get(k, "") for k in DETAIL_CSV_FIELDS}
        )


def save_card_raw(cf: str, full_text: str, html=None):
    safe = re.sub(r"[^A-Za-z0-9_-]", "_", cf or f"unknown_{int(time.time()*1000)}")
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    (RAW_DIR / f"{safe}.txt").write_text(full_text, encoding="utf-8")
    if html is not None:
        (RAW_DIR / f"{safe}.html").write_text(html, encoding="utf-8")


def save_detail_html(cf: str, html: str):
    safe = re.sub(r"[^A-Za-z0-9_-]", "_", cf or f"unknown_{int(time.time()*1000)}")
    RAW_DETTAGLIO_DIR.mkdir(parents=True, exist_ok=True)
    (RAW_DETTAGLIO_DIR / f"{safe}.html").write_text(html, encoding="utf-8")


def ensure_errori_csv():
    if not ERRORI_CSV.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with ERRORI_CSV.open("w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=ERRORI_FIELDS, delimiter=";").writeheader()


def save_errore(keyword, pagina, posizione, ragione, cf, errore):
    ensure_errori_csv()
    with ERRORI_CSV.open("a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=ERRORI_FIELDS, delimiter=";").writerow({
            "keyword": keyword, "pagina": pagina, "posizione": posizione,
            "ragione_sociale": ragione, "codice_fiscale": cf, "errore": str(errore)[:300],
        })


def ensure_toggle_csv_header():
    if not TOGGLE_CSV_PATH.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with TOGGLE_CSV_PATH.open("w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=TOGGLE_CSV_FIELDS, delimiter=";").writeheader()


def append_toggle_row(cf: str, toggle_flags: dict):
    '''Salva SUBITO i toggle (SI/NO/elenco) mentre la pagina e' ancora
    viva: non sono recuperabili in seguito dall'HTML salvato, perche' le
    immagini che li disegnano puntano a un endpoint legato alla sessione
    del browser (fuori sessione non si caricano piu').'''
    ensure_toggle_csv_header()
    with TOGGLE_CSV_PATH.open("a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=TOGGLE_CSV_FIELDS, delimiter=";").writerow(
            {"codice_fiscale": cf, **{k: toggle_flags.get(k, "") for k in TOGGLE_CSV_FIELDS if k != "codice_fiscale"}}
        )


def load_toggle_flags() -> dict:
    '''codice_fiscale -> dict dei toggle gia' catturati.'''
    flags = {}
    if TOGGLE_CSV_PATH.exists():
        with TOGGLE_CSV_PATH.open(newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f, delimiter=";"):
                cf = r.get("codice_fiscale")
                if cf:
                    flags[cf] = {k: v for k, v in r.items() if k != "codice_fiscale"}
    return flags


In [9]:
# --------------------------------------------------------------------------- #
# Helper di ispezione: prima card + suo dettaglio, senza salvare nulla
# --------------------------------------------------------------------------- #
def inspect(driver, keyword: str):
    do_search(driver, keyword)
    cards = get_cards(driver)
    print(f"[inspect] keyword={keyword!r}  total={total_results(driver)}  "
          f"cards_on_page={len(cards)}")
    if not cards:
        return

    row, text = parse_card(cards[0], keyword)
    print("[inspect] card:")
    for k, v in row.items():
        print(f"    {k:26}: {v}")

    html, url, opened_new_tab, toggle_flags = click_apri_dettaglio(driver, cards[0])
    if html is None:
        print("[inspect] impossibile aprire il dettaglio (click senza risultato)")
        return
    campi = estrai_campi_dettaglio(html)
    campi.update(toggle_flags)
    print(f"\n[inspect] dettaglio (url={url}):")
    for k, v in campi.items():
        anteprima = (v[:150] + "...") if isinstance(v, str) and len(v) > 150 else v
        print(f"    {k:32}: {anteprima}")

    if not opened_new_tab:
        torna_ai_risultati(driver, first_card_fingerprint(driver), keyword, 1)


In [10]:
# --------------------------------------------------------------------------- #
# Scrape completo: card + dettaglio, ripartibile, non si ferma sugli errori
# --------------------------------------------------------------------------- #
def load_keywords(cli_keywords=None):
    if cli_keywords:
        return cli_keywords
    return [l.strip() for l in KEYWORDS_FILE.read_text(encoding="utf-8").splitlines()
            if l.strip() and not l.startswith("#")]

def normalizza_cf(cf: str) -> str:
    '''Il CSV esterno a volte perde gli zeri iniziali del codice fiscale
    (es. file aperto/salvato in Excel, che tratta il campo come numero):
    se il codice ha meno di 11 caratteri lo completa con zeri a sinistra
    fino a raggiungere 11.'''
    cf = (cf or "").strip()
    if cf and len(cf) < 11:
        cf = cf.zfill(11)
    return cf

def carica_codici_fiscali_da_csv(path, colonna: str = "codice fiscale",
                                  sep: str = "|", encoding: str = "cp1252"):
    '''Legge i codici fiscali direttamente da un CSV esterno (es.
    l'estrazione startup_06072026.csv), senza bisogno di un file
    codici_fiscali.txt a parte. Il file del progetto usa "|" come
    separatore e encoding cp1252 (i caratteri accentati sono codificati
    cosi', non in UTF-8); i nomi colonna hanno spesso spazi finali, per
    questo li ripulisco prima di cercare "colonna".'''
    import pandas as pd
    df = pd.read_csv(path, sep=sep, dtype=str, engine="python", encoding=encoding)
    df.columns = [c.strip() for c in df.columns]
    if colonna not in df.columns:
        raise ValueError(f"colonna {colonna!r} non trovata. Colonne disponibili: {list(df.columns)}")
    return [normalizza_cf(cf) for cf in df[colonna].astype(str).str.strip().tolist() if cf]


def load_codici_fiscali(cli_cf=None):
    if cli_cf:
        return [normalizza_cf(cf) for cf in cli_cf]
    if CF_CSV_PATH:
        return carica_codici_fiscali_da_csv(CF_CSV_PATH, colonna=CF_CSV_COLONNA)
    if not CF_FILE.exists():
        return []
    return [normalizza_cf(l.strip()) for l in CF_FILE.read_text(encoding="utf-8").splitlines()
            if l.strip() and not l.startswith("#")]


def scrape(driver, keywords, max_pages=0, save_card_html=False,
           fetch_details=True, pause=1.0, restart_every=0, headless=True):
    '''Fase "raccolta": card + HTML del dettaglio + toggle (questi ultimi
    catturati subito, dal vivo, perche' non recuperabili in seguito). NON
    estrae piu' i campi di testo qui: quello lo fa, offline e senza
    browser, estrai_dettagli_da_html_salvati() -- rilanciabile quante
    volte si vuole, anche dopo aver corretto/migliorato l'estrazione,
    senza dover riscaricare nulla.'''
    ensure_csv_header()
    ensure_toggle_csv_header()
    seen = load_seen()
    total_new = 0
    print(f"[start] {len(seen)} aziende gia' salvate; {len(keywords)} keyword; "
          f"fetch_details={fetch_details}")

    for ki, kw in enumerate(keywords, 1):
        print(f"\n=== [{ki}/{len(keywords)}] keyword: {kw!r} ===")
        try:
            do_search(driver, kw)
        except Exception as e:
            print(f"  [errore ricerca] {e}")
            continue

        total = total_results(driver)
        pages = math.ceil(total / 10) if total else 0
        if max_pages:
            pages = min(pages, max_pages) if pages else max_pages
        print(f"  {total} risultati (~{math.ceil(total/10) if total else 0} pagine),"
              f" fino a {pages or '?'} pagine")

        try:
            page = 1
            kw_new = 0
            while True:
                cards = get_cards(driver)
                for idx in range(len(cards)):
                    cards = get_cards(driver)  # rilegge per evitare riferimenti stale
                    if idx >= len(cards):
                        break
                    try:
                        row, text = parse_card(cards[idx], kw)
                        card_html = cards[idx].get_attribute("outerHTML") if save_card_html else None
                    except StaleElementReferenceException:
                        continue
                    except Exception as e:
                        print(f"    [p{page} #{idx}] errore parsing card: {e}")
                        continue

                    cf = row["codice_fiscale"]
                    if cf and cf in seen:
                        continue

                    if fetch_details:
                        fingerprint_pagina = first_card_fingerprint(driver)
                        try:
                            html, url, opened_new_tab, toggle_flags = click_apri_dettaglio(driver, cards[idx])
                        except Exception as e:
                            html, url, opened_new_tab, toggle_flags = None, None, False, {}
                            print(f"    [p{page} #{idx}] errore click dettaglio: {e}")

                        if html is None:
                            save_errore(kw, page, idx, row["ragione_sociale"], cf, "click senza risultato")
                        else:
                            # Salvo SUBITO, mentre la pagina e' ancora viva: l'HTML
                            # (per i campi di testo, estraibili offline in seguito)
                            # e i toggle (SI/NO/elenco), che invece vanno catturati
                            # ORA perche' dipendono da immagini legate alla sessione
                            # del browser e non sono recuperabili da un HTML salvato.
                            try:
                                save_detail_html(cf, html)
                                append_toggle_row(cf, toggle_flags)
                            except Exception as e:
                                save_errore(kw, page, idx, row["ragione_sociale"], cf, f"salvataggio: {e}")

                        if html is not None and not opened_new_tab:
                            tornato = torna_ai_risultati(driver, fingerprint_pagina, kw, page)
                            if not tornato:
                                print(f"    [p{page} #{idx}] 'indietro' non ha funzionato, rifatta la ricerca")
                        elif html is None:
                            try:
                                do_search(driver, kw)
                                for _ in range(page - 1):
                                    if not go_to_next_page(driver):
                                        break
                            except Exception as e:
                                print(f"    [p{page} #{idx}] impossibile ripristinare la ricerca "
                                      f"dopo click fallito: {e}")

                    append_row(row)
                    save_card_raw(cf, text, card_html)
                    if cf:
                        seen.add(cf)
                    total_new += 1
                    kw_new += 1
                    print(f"    [p{page} #{idx}] {row['ragione_sociale'] or cf} OK (totale nuove {total_new})")

                    if pause:
                        time.sleep(pause)

                    if fetch_details and restart_every and total_new and total_new % restart_every == 0:
                        print(f"  [batch] riavvio il browser dopo {total_new} aziende...")
                        driver.quit()
                        driver = build_driver(headless=headless)
                        try:
                            do_search(driver, kw)
                            for _ in range(page - 1):
                                if not go_to_next_page(driver):
                                    break
                        except Exception as e:
                            print(f"  [batch] ricerca dopo riavvio fallita: {e}")

                print(f"  pagina {page}: +{kw_new} nuove finora (totale nuove {total_new})")

                if max_pages and page >= max_pages:
                    break
                if not go_to_next_page(driver):
                    break
                page += 1

            print(f"  fatto {kw!r}: +{kw_new} nuove")
        except Exception as e:
            print(f"  [errore imprevisto su {kw!r}] {e} -- passo alla keyword successiva")
            continue

    print(f"\n[finito] {total_new} nuove aziende. CSV: {CSV_PATH}")
    return driver



# --------------------------------------------------------------------------- #
# Modalita' alternativa: cerca DIRETTAMENTE per codice fiscale invece che
# per keyword. Piu' mirata (un CF = un risultato esatto, niente
# paginazione) e piu' veloce: non serve MAI tornare indietro ai risultati,
# perche' la prossima azienda parte comunque da un nuovo do_search(),
# quindi il problema del "back" che spesso falliva qui non si pone.
# --------------------------------------------------------------------------- #
def scrape_da_codici_fiscali(driver, lista_cf, pause: float = 1.0):
    ensure_csv_header()
    ensure_toggle_csv_header()
    seen = load_seen()
    da_fare = [cf for cf in lista_cf if cf and cf not in seen]
    print(f"[start] {len(seen)} aziende gia' salvate; {len(da_fare)}/{len(lista_cf)} "
          f"codici fiscali da cercare")

    totale_nuove = 0
    n_non_trovate = 0        # ricerca senza nessuna card per quel CF
    n_in_errore = 0          # CF non corrisponde oppure eccezione imprevista
    n_dettaglio_fallito = 0  # trovata e salvata in companies.csv, ma il
                          # dettaglio (HTML/toggle) non e' stato salvato
    for i, cf in enumerate(da_fare, 1):
        print(f"  [{i}/{len(da_fare)}] {cf}...", end=" ")
        try:
            do_search(driver, cf)
            cards = get_cards(driver)
            if not cards:
                print("NON TROVATA")
                save_errore("(lista CF)", 0, 0, "", cf, "non trovata cercando per CF")
                n_non_trovate += 1
                if pause:
                    time.sleep(pause)
                continue

            _, cf_trovato = get_card_identity(cards[0])
            if cf_trovato != cf:
                print(f"CF non corrisponde ({cf_trovato!r})")
                save_errore("(lista CF)", 0, 0, "", cf, f"ricerca CF: trovato {cf_trovato!r} invece")
                n_in_errore += 1
                if pause:
                    time.sleep(pause)
                continue

            row, text = parse_card(cards[0], "(lista CF)")

            try:
                html, url, opened_new_tab, toggle_flags = click_apri_dettaglio(driver, cards[0])
            except Exception as e:
                html, url, opened_new_tab, toggle_flags = None, None, False, {}
                print(f"errore click dettaglio: {e}", end=" ")

            if html is None:
                save_errore("(lista CF)", 0, 0, row["ragione_sociale"], cf, "click senza risultato")
                n_dettaglio_fallito += 1    
            else:
                try:
                    save_detail_html(cf, html)
                    append_toggle_row(cf, toggle_flags)
                except Exception as e:
                    save_errore("(lista CF)", 0, 0, row["ragione_sociale"], cf, f"salvataggio: {e}")
                    n_dettaglio_fallito += 1
            # Non serve "tornare ai risultati": la prossima azienda fa comunque
            # una ricerca nuova (do_search parte con driver.get(SEARCH_URL)).

            append_row(row)
            save_card_raw(cf, text)
            seen.add(cf)
            totale_nuove += 1
            print(f"OK ({row['ragione_sociale']}) totale nuove {totale_nuove}")
        except Exception as e:
            print(f"errore imprevisto: {e}")
            save_errore("(lista CF)", 0, 0, "", cf, f"errore: {e}")
            n_in_errore += 1 

        if pause:
            time.sleep(pause)

    print(f"\n[finito] {totale_nuove} nuove aziende su {len(da_fare)} cercate")
    print()
    print("--- riepilogo ricerca per codice fiscale ---")
    print(f"  trovate (salvate in companies.csv):        {totale_nuove}")
    print(f"    di cui con dettaglio NON salvato "
          f"(vedi errori_estrazione.csv): {n_dettaglio_fallito}")
    print(f"  non trovate sul sito (nessuna card per il CF): {n_non_trovate}")
    print(f"  in errore (CF non corrisponde / eccezione):    {n_in_errore}")
    print(f"  totale cercate in questa sessione:             {len(da_fare)} "
          f"(= {totale_nuove} + {n_non_trovate} + {n_in_errore})")
    return driver

In [11]:
# --------------------------------------------------------------------------- #
# Fase "estrazione" (offline, NESSUN browser, ripetibile all'infinito):
# legge tutti gli HTML gia' salvati in data/raw_dettaglio/ e i toggle gia'
# catturati in data/toggle_flags.csv, e ricostruisce dettagli_estratti.csv.
# Puoi rilanciarla ogni volta che correggi/migliori estrai_campi_dettaglio,
# senza mai dover riscaricare nulla dal sito.
# --------------------------------------------------------------------------- #
def estrai_dettagli_da_html_salvati(sovrascrivi: bool = False):
    ensure_detail_csv_header()

    gia_fatti = set()
    if DETAIL_CSV_PATH.exists() and not sovrascrivi:
        with DETAIL_CSV_PATH.open(newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f, delimiter=";"):
                if r.get("codice_fiscale"):
                    gia_fatti.add(r["codice_fiscale"])

    anagrafica = {}  # cf -> (ragione_sociale, keyword)
    if CSV_PATH.exists():
        with CSV_PATH.open(newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f, delimiter=";"):
                cf = r.get("codice_fiscale")
                if cf:
                    anagrafica[cf] = (r.get("ragione_sociale", ""), r.get("keyword", ""))

    toggle_flags = load_toggle_flags()

    if sovrascrivi:
        ensure_csv_header()
        with DETAIL_CSV_PATH.open("w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=DETAIL_CSV_FIELDS, delimiter=";").writeheader()

    n_ok, n_saltati = 0, 0
    for path in sorted(RAW_DETTAGLIO_DIR.glob("*.html")):
        cf = path.stem
        if cf in gia_fatti:
            continue
        try:
            html = path.read_text(encoding="utf-8")
            campi = estrai_campi_dettaglio(html)
            campi.update(toggle_flags.get(cf, {}))
            ragione, kw = anagrafica.get(cf, ("", ""))
            detail_row = {
                "keyword": kw, "ragione_sociale": ragione, "codice_fiscale": cf,
                "url": "", **campi,
            }
            append_detail_row(detail_row)
            n_ok += 1
        except Exception as e:
            print(f"  [estrai] errore su {path.name}: {e}")
            n_saltati += 1

    print(f"[estrai] {n_ok} dettagli scritti, {n_saltati} falliti, "
          f"da {len(list(RAW_DETTAGLIO_DIR.glob('*.html')))} HTML totali "
          f"in {RAW_DETTAGLIO_DIR}")


# --------------------------------------------------------------------------- #
# Fase "ripara" (dal vivo, browser necessario, SOLO per chi non ha nemmeno
# l'HTML salvato): aziende in companies.csv senza un file in
# raw_dettaglio/ (il click era fallito la prima volta). Le cerca per
# codice fiscale, riapre il dettaglio, salva HTML + toggle come nella
# scrape normale. Dopo aver rilanciato questa funzione, rilancia
# estrai_dettagli_da_html_salvati() per completare il CSV.
# --------------------------------------------------------------------------- #
def ripara_dettagli_mancanti(driver, max_aziende: int = 0, pause: float = 1.0):
    ensure_toggle_csv_header()

    salvati = {p.stem for p in RAW_DETTAGLIO_DIR.glob("*.html")}

    aziende = []
    if CSV_PATH.exists():
        with CSV_PATH.open(newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f, delimiter=";"):
                cf = r.get("codice_fiscale")
                if cf and cf not in salvati:
                    aziende.append(r)

    print(f"[ripara] {len(aziende)} aziende senza HTML di dettaglio su {len(salvati) + len(aziende)} totali")
    if max_aziende:
        aziende = aziende[:max_aziende]

    ok, falliti = 0, 0
    for i, r in enumerate(aziende, 1):
        cf = r["codice_fiscale"]
        kw = r.get("keyword", "")
        ragione = r.get("ragione_sociale", "")
        print(f"  [{i}/{len(aziende)}] {ragione or cf}...", end=" ")

        try:
            do_search(driver, cf)
            cards = get_cards(driver)
            if not cards:
                print("NON TROVATA cercando per CF")
                save_errore(kw, 0, 0, ragione, cf, "ripara: non trovata cercando per CF")
                falliti += 1
                continue
            _, cf_trovato = get_card_identity(cards[0])
            if cf_trovato != cf:
                print(f"CF non corrisponde ({cf_trovato!r} != {cf!r})")
                save_errore(kw, 0, 0, ragione, cf, f"ripara: CF non corrisponde ({cf_trovato})")
                falliti += 1
                continue

            html, url, opened_new_tab, toggle_flags = click_apri_dettaglio(driver, cards[0])
            if html is None:
                print("click dettaglio fallito di nuovo")
                save_errore(kw, 0, 0, ragione, cf, "ripara: click senza risultato")
                falliti += 1
                continue

            save_detail_html(cf, html)
            append_toggle_row(cf, toggle_flags)
            ok += 1
            print("OK (HTML + toggle salvati)")
        except Exception as e:
            print(f"errore: {e}")
            save_errore(kw, 0, 0, ragione, cf, f"ripara: {e}")
            falliti += 1

        if pause:
            time.sleep(pause)

    print(f"\n[ripara] fatto: {ok} recuperati, {falliti} falliti. "
          f"Ora lancia estrai_dettagli_da_html_salvati() per completare il CSV.")
    return driver

## Parametri di esecuzione

Prova prima in piccolo (es. `MAX_PAGES = 1` su una sola keyword) prima di
lanciare la scansione completa su migliaia di aziende.


In [ ]:
KEYWORDS = None        # es. ["energia"] oppure None per usare keywords.txt
CERCA_PER_CF = True     # True: ignora KEYWORDS, cerca direttamente i codici
                          # fiscali (piu' veloce e mirato: niente paginazione,
                          # niente "torna ai risultati")
LISTA_CF = None          # es. ["12345678901", ...]. Ha priorita' su CF_CSV_PATH.
# CF_CSV_PATH = r"C:\path\to\your\startup_export.csv"

                          # Se valorizzato (e LISTA_CF e' None), i codici
                          # fiscali vengono letti direttamente da qui invece
                          # che da codici_fiscali.txt. Metti None per tornare
                          # al file di testo.
CF_CSV_COLONNA = "codice fiscale"  # nome della colonna nel CSV
NO_HEADLESS = False      # True per vedere il browser mentre lavora
INSPECT = False         # True: test rapido, solo prima card + suo dettaglio
MAX_PAGES = 0           # 0 = tutte le pagine (per il test lascia 1)
SAVE_CARD_HTML = False  # True per salvare anche l'HTML grezzo di ogni card
FETCH_DETAILS = True    # True per aprire ed estrarre anche il dettaglio
PAUSE = 1.0             # pausa (secondi) tra un'azienda e l'altra
RESTART_EVERY = 300     # riavvia il browser ogni tot aziende (0 = mai)

RIPARA_MANCANTI = False  # True: (browser) cerca dal vivo le aziende senza
                          # HTML di dettaglio salvato e lo scarica
RIPARA_MAX = 0            # 0 = tutte le mancanti, altrimenti limite per test

ESTRAI_OFFLINE = False    # True: (NESSUN browser) rilegge tutti gli HTML gia'
                          # salvati + i toggle gia' catturati e ricostruisce
                          # dettagli_estratti.csv. Rilanciabile quante volte
                          # vuoi, anche solo per aggiungere un campo nuovo.
ESTRAI_SOVRASCRIVI = False  # True: ributta giu' tutto dettagli_estratti.csv
                            # e lo ricrea da zero (utile dopo aver aggiunto
                            # un campo a tutte le aziende gia' fatte)


In [20]:
RAW_DIR.mkdir(parents=True, exist_ok=True)

if ESTRAI_OFFLINE:
    # Nessun browser: rilegge solo gli HTML/toggle gia' salvati su disco.
    estrai_dettagli_da_html_salvati(sovrascrivi=ESTRAI_SOVRASCRIVI)
else:
    driver = build_driver(headless=not NO_HEADLESS)
    try:
        if RIPARA_MANCANTI:
            driver = ripara_dettagli_mancanti(driver, max_aziende=RIPARA_MAX, pause=PAUSE)
        elif CERCA_PER_CF:
            lista_cf = load_codici_fiscali(LISTA_CF)
            driver = scrape_da_codici_fiscali(driver, lista_cf, pause=PAUSE)
        elif INSPECT:
            keywords = load_keywords(KEYWORDS)
            inspect(driver, keywords[0])
        else:
            keywords = load_keywords(KEYWORDS)
            driver = scrape(driver, keywords, max_pages=MAX_PAGES, save_card_html=SAVE_CARD_HTML,
                             fetch_details=FETCH_DETAILS, pause=PAUSE, restart_every=RESTART_EVERY,
                             headless=not NO_HEADLESS)
    except KeyboardInterrupt:
        print("\n[interrotto] risultati parziali gia' salvati su disco.")
    except Exception:
        traceback.print_exc()
    finally:
        driver.quit()



[start] 4 aziende gia' salvate; 53/57 codici fiscali da cercare
  [1/53] 01884130939... NON TROVATA
  [2/53] 01955970494... NON TROVATA
  [3/53] 02664560030... NON TROVATA
  [4/53] 03006270304... NON TROVATA
  [5/53] 03048670644... NON TROVATA
  [6/53] 03738210545... NON TROVATA
  [7/53] 03896251208... NON TROVATA
  [8/53] 03921570366... NON TROVATA
  [9/53] 03926810130...   [do_search] tentativo 1/3 fallito (timeout 45s)
  [do_search] tentativo 2/3 fallito (timeout 90s)
NON TROVATA
  [10/53] 03968760136... NON TROVATA
  [11/53] 03971361203... NON TROVATA
  [12/53] 04480070160... NON TROVATA
  [13/53] 04564020610... NON TROVATA
  [14/53] 06860340824... NON TROVATA
  [15/53] 08587960728... NON TROVATA
  [16/53] 11130820969... NON TROVATA
  [17/53] 11180210962... NON TROVATA
  [18/53] 11196420969... NON TROVATA
  [19/53] 11230860964... NON TROVATA
  [20/53] 11242420963...   [do_search] tentativo 1/3 fallito (timeout 45s)
  [do_search] tentativo 2/3 fallito (timeout 90s)
NON TROVATA
  [21